In [1]:
import sys; sys.path.append("../..")
from donnees import charger_compteurs, parties, seuil, ecrire_predictions, COLONNES_E

In [2]:
df = charger_compteurs()
df.head()

,block_id,label,split,E1,E2,E3,E4,E5,E6,E7,...,E26,E27,E28,E29,E30,n_warn,n_error,length,n_unique,duration_s
0,blk_-1608999687919862906,0,train,0,0,203,0,10,7,0,...,10,0,0,0,0,0,0,269,12,3802
1,blk_7503483334202473044,0,train,0,2,1,0,3,0,0,...,3,0,0,0,0,0,0,22,9,3802
2,blk_-3544583377289625738,1,train,0,0,203,0,3,0,0,...,3,0,0,0,0,1,0,223,9,3797
3,blk_-9073992586687739851,0,train,0,3,0,0,3,0,0,...,3,0,0,0,0,0,0,22,8,50448
4,blk_7854771516489510256,0,train,0,3,1,15,3,0,0,...,3,0,0,0,0,15,0,38,10,50583


In [3]:
print(df.shape)                       # attendu : (575061, nombre_de_colonnes)
print(df["split"].value_counts())     # train 402542, val 86259, test 86260
print(df.groupby("split")["label"].mean())   # part d'anomalies : 3.26%, 3.02%, 1.29%

(575061, 38)
split
train    402542
test      86260
val       86259
Name: count, dtype: int64
split
test     0.012938
train    0.032580
val      0.030223
Name: label, dtype: float64


In [4]:
# imports et préparation
import numpy as np
import time
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score

p = parties(df)

X_train      = p["train_normal"][COLONNES_E].values
X_val        = p["val"][COLONNES_E].values
y_val        = p["val"]["label"].values
X_val_normal = p["val_normal"][COLONNES_E].values
X_test       = p["test"][COLONNES_E].values

In [5]:
print(X_train.shape, X_val.shape, X_val_normal.shape, X_test.shape)

(389427, 30) (86259, 30) (83652, 30) (86260, 30)


In [6]:
# la fonction de score
def score(model, X):
    return -model.score_samples(X)

In [12]:
resultats = {}
for n in [100, 200]:
    for ms in [1024, 4096, 16384, 65536]:
        model = IsolationForest(n_estimators=n, max_samples=ms, random_state=42, n_jobs=-1)
        model.fit(X_train)
        s = seuil(score(model, X_val_normal))
        f1 = f1_score(y_val, score(model, X_val) > s)
        resultats[(n, ms)] = f1
        print(n, ms, round(f1, 4))

meilleur = max(resultats, key=resultats.get)
print("Meilleur réglage :", meilleur)

100 1024 0.336
100 4096 0.3379
100 16384 0.3451
100 65536 0.374
200 1024 0.336
200 4096 0.3374
200 16384 0.3456
200 65536 0.374
Meilleur réglage : (100, 65536)


In [13]:
# entraînement final et fichier de prédictions
t0 = time.time()
model = IsolationForest(n_estimators=meilleur[0], max_samples=meilleur[1],
                        random_state=42, n_jobs=-1).fit(X_train)
temps_train = (time.time() - t0) / 60
print("Temps d'entraînement (minutes) :", round(temps_train, 2))

s = seuil(score(model, X_val_normal))

t0 = time.time()
scores_test = score(model, X_test)
temps_pred = time.time() - t0
print("Temps de prédiction (secondes) :", round(temps_pred, 1))

ecrire_predictions(p["test"], scores_test, s, "../../results/isolation_forest_test.csv")

Temps d'entraînement (minutes) : 0.02
Temps de prédiction (secondes) : 0.3


,block_id,score,pred
0,blk_3458545012440520721,0.475655,0
1,blk_-1160437354072569825,0.475655,0
2,blk_-8961379332018832424,0.475655,0
3,blk_-7458466477853961957,0.475655,0
4,blk_3070076663445833576,0.475655,0
...,...,...,...
86255,blk_1019720114020043203,0.348552,0
86256,blk_-2683116845478050414,0.348552,0
86257,blk_5595059397348477632,0.348552,0
86258,blk_1513937873877967730,0.348552,0


In [14]:
import pandas as pd
predictions = pd.read_csv("../../results/isolation_forest_test.csv")
print(predictions["pred"].value_counts())

pred
0    85920
1      340
Name: count, dtype: int64


In [ ]:
# Sur le test, il y a 1,29 % d'anomalies (1 116 blocs sur 86 260).
# modèle lève 0,39 % d'alertes (340 blocs)

In [15]:
print("Seuil :", s)

print("Score minimum :", scores_test.min())
print("Score maximum :", scores_test.max())

Seuil : 0.538580000666987
Score minimum : 0.3485517078499684
Score maximum : 0.8863786412099263


In [17]:
print("Pourcentage d'anomalies prédites :",
      round((predictions["pred"] == 1).mean() * 100, 2), "%")

Pourcentage d'anomalies prédites : 0.39 %


In [18]:
import pandas as pd
chk = pd.read_csv("../../results/isolation_forest_test.csv")
print(chk.shape)       # (86260, 3)
print(chk.head())      # block_id, score, pred

(86260, 3)
                   block_id     score  pred
0   blk_3458545012440520721  0.475655     0
1  blk_-1160437354072569825  0.475655     0
2  blk_-8961379332018832424  0.475655     0
3  blk_-7458466477853961957  0.475655     0
4   blk_3070076663445833576  0.475655     0
